In [1]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import keras_tuner as kt
import tensorboard

I0000 00:00:1785488800.512024    3647 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785488800.525265    3647 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785488802.035955    3647 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785488804.718727    3647 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [3]:
img_size = (224,224)
batch_size = 32

In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(factor=0.15, value_range=(0.0, 1.0)),
])

E0000 00:00:1785488815.644506    3647 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [5]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [6]:
def preprocess_train(image_path, label):
    image, label = preprocess_image(image_path, label)
    image = data_augmentation(image)
    return image, label

In [7]:
def preprocess_test(image_path, label):
    image, label = preprocess_image(image_path, label)
    return image, label

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset = (
    train_dataset
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [9]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset = (
    test_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [10]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset = (
    val_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [12]:
#batch size 64

In [11]:
batch_size_64 = 64

In [50]:
train_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset_64 = (
    train_dataset_64
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [51]:
test_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset_64 = (
    test_dataset_64
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [57]:
val_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset_64 = (
    val_dataset_64
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [53]:
image,label = next(iter(test_dataset_64))

print(image.shape)

(64, 224, 224, 3)


In [54]:
image,label = next(iter(test_dataset))

print(image.shape)

(32, 224, 224, 3)


In [56]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [91]:
'''CNN BASELINE WITH EARLY STOP'''

early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [15]:
Bmodel_cnn_earlystop = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
Bmodel_cnn_earlystop .summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
Bmodel_cnn_earlystop.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [17]:
history = Bmodel_cnn_earlystop.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/backend/tensorflow/nn.py:1402: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 817ms/step - accuracy: 0.2388 - loss: 1.9554 - val_accuracy: 0.0346 - val_loss: 2.0064
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 817ms/step - accuracy: 0.2414 - loss: 1.8384 - val_accuracy: 0.0745 - val_loss: 2.0603
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 799ms/step - accuracy: 0.2876 - loss: 1.7675 - val_accuracy: 0.2063 - val_loss: 1.8865
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 792ms/step - accuracy: 0.2731 - loss: 1.7143 - val_accuracy: 0.2349 - val_loss: 1.7927
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 175s 773ms/step - accuracy: 0.3814 - loss: 1.6275 - val_accuracy: 0.2355 - val_loss: 1.8308
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 803ms/step - accuracy: 0.4326 - loss: 1.5687 - val_accuracy: 0.3759 - val_loss: 1.6705
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 190s 840ms/step - accuracy: 0.4586 - loss: 1.5393 - val_accuracy: 0.4478 - val_loss: 1.3644
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 186s 822ms/step - accuracy: 0.4256 - loss: 1.49

In [18]:
test_loss,test_accuracy = Bmodel_cnn_earlystop.evaluate(test_dataset)
print(f'test accuracy : {test_accuracy}')
print(f'test loss : {test_loss}')

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 169ms/step - accuracy: 0.4804 - loss: 1.2556
test accuracy : 0.4803725779056549
test loss : 1.2556341886520386


In [21]:
Bmodel_cnn_earlystop.save(r'../models/b_cnn_earlystop.keras')

In [ ]:
'''learning rate reshedule'''

In [22]:
Bmodel_cnn_learning = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
Bmodel_cnn_learning .summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
Bmodel_cnn_learning.compile(optimizer=optimizer,
              loss=tf.keras.losses.SparseCategoricalCrossentropy,
              metrics=['accuracy'])

In [25]:
history = Bmodel_cnn_learning.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict
)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 804ms/step - accuracy: 0.3096 - loss: 1.9698 - val_accuracy: 0.2402 - val_loss: 1.8965
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 186s 824ms/step - accuracy: 0.2341 - loss: 1.9067 - val_accuracy: 0.0200 - val_loss: 2.1816
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 821ms/step - accuracy: 0.1280 - loss: 1.9114 - val_accuracy: 0.2242 - val_loss: 1.7210
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 812ms/step - accuracy: 0.0897 - loss: 1.9658 - val_accuracy: 0.0432 - val_loss: 2.0427
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 813ms/step - accuracy: 0.0913 - loss: 1.9288 - val_accuracy: 0.0246 - val_loss: 2.0332
Epoch 6/10
106/220 ━━━━━━━━━━━━━━━━━━━━ 1:34 832ms/step - accuracy: 0.1197 - loss: 1.8425

KeyboardInterrupt: 

In [57]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [29]:
Bmodel_cnn_learning = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
Bmodel_cnn_learning .summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
Bmodel_cnn_learning.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [36]:
history = Bmodel_cnn_learning.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[lr_scheduler]
)

Epoch 1/5


220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 805ms/step - accuracy: 0.2821 - loss: 1.9408 - val_accuracy: 0.4365 - val_loss: 1.7284 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 821ms/step - accuracy: 0.2126 - loss: 1.8373 - val_accuracy: 0.2156 - val_loss: 1.9093 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 728ms/step - accuracy: 0.2330 - loss: 1.7822
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 176s 778ms/step - accuracy: 0.2330 - loss: 1.7822 - val_accuracy: 0.2601 - val_loss: 1.7615 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 818ms/step - accuracy: 0.3013 - loss: 1.7194 - val_accuracy: 0.4165 - val_loss: 1.5262 - learning_rate: 5.0000e-04
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 818ms/step - accuracy: 0.4607 - loss: 1.6292 - val_accuracy: 0.5476 - val_loss: 1.3041 - learning_rate: 5.0000e-04


In [41]:
test_loss, test_accuracy = Bmodel_cnn_learning.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 200ms/step - accuracy: 0.5462 - loss: 1.3054


In [42]:
Bmodel_cnn_learning.save(r'../models/b_cnn_learingrate.keras')

In [ ]:
'''optimizer keras sdg'''

In [49]:
Bmodel_cnn_sgd = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
Bmodel_cnn_sgd .summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
Bmodel_cnn_sgd.compile(optimizer=tf.keras.optimizers.SGD(),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [51]:
history = Bmodel_cnn_sgd.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict,
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 158s 696ms/step - accuracy: 0.2356 - loss: 1.9408 - val_accuracy: 0.2621 - val_loss: 1.9263
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 718ms/step - accuracy: 0.3757 - loss: 1.8880 - val_accuracy: 0.5649 - val_loss: 1.3310
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 167s 738ms/step - accuracy: 0.4063 - loss: 1.8138 - val_accuracy: 0.6081 - val_loss: 1.2378
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 720ms/step - accuracy: 0.4012 - loss: 1.7769 - val_accuracy: 0.1657 - val_loss: 1.9459
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 717ms/step - accuracy: 0.4035 - loss: 1.7070 - val_accuracy: 0.1637 - val_loss: 1.7495
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 172s 759ms/step - accuracy: 0.4108 - loss: 1.6137 - val_accuracy: 0.5875 - val_loss: 1.2034
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 780ms/step - accuracy: 0.4608 - loss: 1.5082 - val_accuracy: 0.4059 - val_loss: 1.3628
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 174s 767ms/step - accuracy: 0.4701 -

In [52]:
test_loss,test_accuracy = Bmodel_cnn_sgd.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 207ms/step - accuracy: 0.4125 - loss: 1.5304


In [53]:
Bmodel_cnn_sgd.save(r'../models/bmodel_cnn_sgd.keras')

In [ ]:
'''adam'''

In [54]:
bmodel_cnn_adam = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
bmodel_cnn_adam.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_18 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
bmodel_cnn_adam.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [56]:
bmodel_cnn_adam.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 191s 841ms/step - accuracy: 0.2675 - loss: 1.9916 - val_accuracy: 0.0186 - val_loss: 2.0250
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 819ms/step - accuracy: 0.1417 - loss: 1.9203 - val_accuracy: 0.1537 - val_loss: 1.5635
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 838ms/step - accuracy: 0.1761 - loss: 1.9096 - val_accuracy: 0.0672 - val_loss: 1.7896
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 802ms/step - accuracy: 0.1816 - loss: 1.9020 - val_accuracy: 0.0293 - val_loss: 2.0910
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 807ms/step - accuracy: 0.1895 - loss: 1.8585 - val_accuracy: 0.0213 - val_loss: 2.1425
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 190s 841ms/step - accuracy: 0.2975 - loss: 1.8043 - val_accuracy: 0.3999 - val_loss: 1.5088
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 816ms/step - accuracy: 0.3220 - loss: 1.7046 - val_accuracy: 0.1470 - val_loss: 2.0651
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 174s 771ms/step - accuracy: 0.3366 - loss: 1.68

In [57]:
test_loss,test_accuracy = bmodel_cnn_adam.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 169ms/step - accuracy: 0.4458 - loss: 1.4226


In [58]:
bmodel_cnn_adam.save(r'../models/b_cnn_adam.keras')

In [59]:
bmodel_cnn_RMS = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
bmodel_cnn_RMS.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_21 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [61]:
bmodel_cnn_RMS.compile(optimizer=tf.keras.optimizers.RMSprop(),loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [62]:
bmodel_cnn_RMS.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 838ms/step - accuracy: 0.3456 - loss: 1.9844 - val_accuracy: 0.0652 - val_loss: 3.5153
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 873ms/step - accuracy: 0.3991 - loss: 1.7063 - val_accuracy: 0.6281 - val_loss: 1.0313
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 782ms/step - accuracy: 0.4193 - loss: 1.6379 - val_accuracy: 0.4371 - val_loss: 1.4429
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 774ms/step - accuracy: 0.4276 - loss: 1.5972 - val_accuracy: 0.5981 - val_loss: 1.1063
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 790ms/step - accuracy: 0.4460 - loss: 1.5875 - val_accuracy: 0.4890 - val_loss: 1.3026
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 176s 781ms/step - accuracy: 0.4624 - loss: 1.5237 - val_accuracy: 0.5103 - val_loss: 1.3274
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 781ms/step - accuracy: 0.4661 - loss: 1.5535 - val_accuracy: 0.5735 - val_loss: 1.1126
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 789ms/step - accuracy: 0.4393 - loss: 1.52

In [63]:
test_loss,test_accuracy = bmodel_cnn_RMS.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 171ms/step - accuracy: 0.6075 - loss: 0.9738


In [64]:
bmodel_cnn_RMS.save(r'../models/bmodel_cnn_RMS.keras')

In [ ]:
'''batch reshaping'''

In [74]:
bmodel_cnn_64 = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
bmodel_cnn_64.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_24 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_25 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_26 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [75]:
bmodel_cnn_64.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [76]:
bmodel_cnn_64.fit(test_dataset_64,validation_data=val_dataset_64,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


24/24 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.3174 - loss: 2.5459 - val_accuracy: 0.3859 - val_loss: 1.7047
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.4145 - loss: 1.8387 - val_accuracy: 0.3513 - val_loss: 1.8189
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.3979 - loss: 1.7437 - val_accuracy: 0.4817 - val_loss: 1.3063
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.3772 - loss: 1.6494 - val_accuracy: 0.5456 - val_loss: 1.2166
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.4178 - loss: 1.4608 - val_accuracy: 0.4757 - val_loss: 1.3157
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5143 - loss: 1.2445 - val_accuracy: 0.4617 - val_loss: 1.4943
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.5216 - loss: 1.1814 - val_accuracy: 0.4271 - val_loss: 1.6019
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5529 - loss: 1.0549 - val_accuracy: 0.5336 - val_loss: 1.3412
Epo

In [78]:
test_loss,test_accuracy = bmodel_cnn_64.evaluate(test_dataset_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 333ms/step - accuracy: 0.7152 - loss: 0.7411


In [79]:
bmodel_cnn_64.save(r'../models/64_batch_base_cnn.keras')

In [ ]:
'''hyper parameter'''

In [47]:
INPUT_SHAPE = (224, 224, 3)
NUM_CLASSES = 7

def build_cnn(hp):

    model = models.Sequential([

        layers.Input(shape=INPUT_SHAPE),

        # Block 1
        layers.Conv2D(
            filters=hp.Choice(
                "filters1",
                values=[32, 64]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        # Block 2
        layers.Conv2D(
            filters=hp.Choice(
                "filters2",
                values=[64, 128]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        # Block 3
        layers.Conv2D(
            filters=hp.Choice(
                "filters3",
                values=[128, 256]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),

        layers.Dense(
            units=hp.Choice(
                "dense_units",
                values=[128, 256]
            ),
            activation="relu"
        ),

        layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2, 0.3, 0.5]
            )
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    '''optimizer_name = 

    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam()

    elif optimizer_name == "sgd":
        optimizer = tf.keras.optimizers.SGD()

    else:
        optimizer = tf.keras.optimizers.RMSprop()'''

    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam", "sgd", "rmsprop"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [50]:
tuner = kt.RandomSearch(
    hypermodel=build_cnn,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="baseline_cnn"
)

In [51]:
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 44m 05s]
val_accuracy: 0.4863606095314026

Best val_accuracy So Far: 0.652694582939148
Total elapsed time: 04h 44m 54s


In [52]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'filters1': 64, 'filters2': 64, 'filters3': 256, 'dense_units': 128, 'dropout': 0.5, 'optimizer': 'rmsprop'}


In [53]:
best_model = tuner.hypermodel.build(best_hp)

In [54]:
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Epoch 1/5


220/220 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.2717 - loss: 2.0255 - val_accuracy: 0.2409 - val_loss: 1.8877
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.3481 - loss: 1.8295 - val_accuracy: 0.5968 - val_loss: 1.1198
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.3754 - loss: 1.7832 - val_accuracy: 0.5429 - val_loss: 1.2314
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.3905 - loss: 1.7254 - val_accuracy: 0.5768 - val_loss: 1.1634
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.3731 - loss: 1.7260 - val_accuracy: 0.5030 - val_loss: 1.2257


In [55]:
test_loss, test_accuracy = best_model.evaluate(test_dataset)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 293ms/step - accuracy: 0.5090 - loss: 1.2230
Test Accuracy : 0.5089820623397827
Test Loss : 1.2229942083358765


In [56]:
best_model.save(r'../models/best_baseline_model.keras')

In [81]:
#deep cnn early stop

early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [85]:
deep_model_earlystop = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model_earlystop.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_37 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_37 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_38 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [86]:
deep_model_earlystop.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [87]:
deep_model_earlystop.fit(test_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/10


47/47 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.4172 - loss: 2.0196 - val_accuracy: 0.6035 - val_loss: 1.3436
Epoch 2/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - accuracy: 0.4571 - loss: 1.8719 - val_accuracy: 0.5729 - val_loss: 1.3423
Epoch 3/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - accuracy: 0.4065 - loss: 1.8163 - val_accuracy: 0.3979 - val_loss: 1.6518
Epoch 4/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.4092 - loss: 1.6957 - val_accuracy: 0.3706 - val_loss: 1.4744
Epoch 5/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.3693 - loss: 1.6516 - val_accuracy: 0.3247 - val_loss: 1.5421
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [88]:
test_loss,test_accuracy = deep_model_earlystop.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 215ms/step - accuracy: 0.5875 - loss: 1.3248


In [89]:
deep_model_earlystop.save(r'../models/deep_cnn_early_stop.keras')

In [ ]:
'''learning rate reshedule'''

In [90]:
deep_cnn_learning = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
deep_cnn_learning .summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_41 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_42 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_43 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [91]:
deep_cnn_learning.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [92]:
history = deep_cnn_learning.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[lr_scheduler]
)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 871ms/step - accuracy: 0.1919 - loss: 2.0167 - val_accuracy: 0.1045 - val_loss: 1.8306 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 862ms/step - accuracy: 0.2241 - loss: 1.8984 - val_accuracy: 0.4657 - val_loss: 1.5029 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 886ms/step - accuracy: 0.2314 - loss: 1.8442 - val_accuracy: 0.1690 - val_loss: 2.0194 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 827ms/step - accuracy: 0.2726 - loss: 1.8270
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 875ms/step - accuracy: 0.2726 - loss: 1.8270 - val_accuracy: 0.1224 - val_loss: 1.8609 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 190s 834ms/step - accuracy: 0.3129 - loss: 1.7583 - val_accuracy: 0.2934 - val_loss: 1.6900 - learning_rate: 5.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 785ms/step - accuracy: 0.3742 -

In [93]:
deep_cnn_learning.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 179ms/step - accuracy: 0.5469 - loss: 1.1476


[1.1476404666900635, 0.5469061732292175]

In [105]:
deep_cnn_learning.save(r'../models/deep_cnn_learning.keras')

In [ ]:
'''deep cnn optmaizer'''

In [94]:
deep_model_SGD = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model_SGD.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_44 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_44 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_45 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_45 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_46 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_46 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_47 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_47 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [97]:
deep_model_SGD.compile(optimizer=tf.keras.optimizers.SGD(),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [99]:
deep_model_SGD.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 214s 959ms/step - accuracy: 0.2518 - loss: 1.9403 - val_accuracy: 0.6028 - val_loss: 1.7188
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 220s 974ms/step - accuracy: 0.3715 - loss: 1.8934 - val_accuracy: 0.4278 - val_loss: 1.5815
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 213s 947ms/step - accuracy: 0.3864 - loss: 1.8452 - val_accuracy: 0.4704 - val_loss: 1.5081
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 969ms/step - accuracy: 0.3996 - loss: 1.7941 - val_accuracy: 0.1344 - val_loss: 1.8561
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 251s 930ms/step - accuracy: 0.4083 - loss: 1.7577 - val_accuracy: 0.4877 - val_loss: 1.4220
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 274s 986ms/step - accuracy: 0.4013 - loss: 1.7235 - val_accuracy: 0.5422 - val_loss: 1.3171
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 961ms/step - accuracy: 0.4041 - loss: 1.7103 - val_accuracy: 0.1171 - val_loss: 2.4041
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.3931 - loss: 1.6573 

In [100]:
test_loss,test_accuracy = deep_model_SGD.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 245ms/step - accuracy: 0.5828 - loss: 1.1433


In [106]:
deep_model_SGD.save(r'../models/deep_cnn_SGD.keras')

In [ ]:
'''adam'''

In [101]:
deep_model_adam = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model_adam.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_48 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_48 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_49 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_49 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_50 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_50 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_51 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_51 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_14 (Flatten)            │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [102]:
deep_model_adam.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(),metrics=['accuracy'])

In [103]:
deep_model_adam.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.1990 - loss: 1.9419 - val_accuracy: 0.0719 - val_loss: 1.9725
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 250s 1s/step - accuracy: 0.2368 - loss: 1.9002 - val_accuracy: 0.1364 - val_loss: 1.6037
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 256s 1s/step - accuracy: 0.1676 - loss: 1.8797 - val_accuracy: 0.2129 - val_loss: 1.6438
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.2605 - loss: 1.7979 - val_accuracy: 0.0951 - val_loss: 1.8395
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 214s 949ms/step - accuracy: 0.2437 - loss: 1.8080 - val_accuracy: 0.1723 - val_loss: 1.8187
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 998ms/step - accuracy: 0.3038 - loss: 1.6993 - val_accuracy: 0.3719 - val_loss: 1.6068
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 250s 1s/step - accuracy: 0.3339 - loss: 1.6365 - val_accuracy: 0.4411 - val_loss: 1.4973
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 209s 922ms/step - accuracy: 0.4623 - loss: 1.4925 - val_accura

In [104]:
test_loss,test_accuracy = deep_model_adam.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 297ms/step - accuracy: 0.5030 - loss: 1.4019


In [107]:
deep_model_adam.save(r'../models/deep_model_adam.keras')

In [ ]:
'''RMSPROP'''

In [108]:
deep_model_RMS = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model_RMS.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_52 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_52 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_53 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_53 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_54 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_54 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_55 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_55 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_15 (Flatten)            │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [1]:
deep_model_RMS.compile(optimizer =tf.keras.optimizers.RMSprop(),loss=tf.keras.losses.SparseCategoricalCrossentropy(),metrics=['accuracy'])

NameError: name 'deep_model_RMS' is not defined

In [110]:
deep_model_RMS.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 960ms/step - accuracy: 0.3413 - loss: 1.9423 - val_accuracy: 0.6055 - val_loss: 1.2356
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.3789 - loss: 1.8566 - val_accuracy: 0.3593 - val_loss: 1.5381
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 978ms/step - accuracy: 0.4142 - loss: 1.7814 - val_accuracy: 0.2309 - val_loss: 2.0462
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.4587 - loss: 1.6442 - val_accuracy: 0.0898 - val_loss: 2.3711
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 983ms/step - accuracy: 0.4637 - loss: 1.5616 - val_accuracy: 0.5622 - val_loss: 1.1685
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 985ms/step - accuracy: 0.4934 - loss: 1.5089 - val_accuracy: 0.3679 - val_loss: 1.5367
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 245s 1s/step - accuracy: 0.4984 - loss: 1.5030 - val_accuracy: 0.5329 - val_loss: 1.1446
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 236s 1s/step - accuracy: 0.5083 - loss: 1.4549 - val_acc

In [111]:
test_loss,test_accuracy = deep_model_RMS.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 274ms/step - accuracy: 0.5782 - loss: 1.1410


In [ ]:
'''batch 64'''

In [24]:
deep_model_64 = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model_64.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
deep_model_64.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [26]:
deep_model_64.fit(train_dataset_64,validation_data=val_dataset_64,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.2711 - loss: 1.9102 - val_accuracy: 0.3906 - val_loss: 1.7027
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.3603 - loss: 1.8175 - val_accuracy: 0.4358 - val_loss: 1.6231
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.3759 - loss: 1.8040 - val_accuracy: 0.3759 - val_loss: 1.5853
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 213s 2s/step - accuracy: 0.3965 - loss: 1.7542 - val_accuracy: 0.2715 - val_loss: 1.8769
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.3838 - loss: 1.6824 - val_accuracy: 0.3353 - val_loss: 1.8234
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.4643 - loss: 1.5476 - val_accuracy: 0.4105 - val_loss: 1.5823
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.3945 - loss: 1.5334 - val_accuracy: 0.4957 - val_loss: 1.4967
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.4751 - loss: 1.5259 - val_accuracy: 0.520

In [28]:
test_loss,test_accuracy = deep_model_64.evaluate(test_dataset_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 374ms/step - accuracy: 0.5788 - loss: 1.0787


In [29]:
deep_model_64.save(r'../models/deep_cnn_64.keras')

In [ ]:
'''hyper parameter'''

In [76]:
INPUT_SHAPE = (224, 224, 3)
NUM_CLASSES = 7

def deep_build_cnn(hp):

    model = models.Sequential([

        layers.Input(shape=INPUT_SHAPE),

        # Block 1
        layers.Conv2D(
            filters=hp.Choice(
                "filters1",
                values=[32, 64]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        # Block 2
        layers.Conv2D(
            filters=hp.Choice(
                "filters2",
                values=[64, 128]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        # Block 3
        layers.Conv2D(
            filters=hp.Choice(
                "filters3",
                values=[128, 256]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),

        layers.Dense(
            units=hp.Choice(
                "dense_units",
                values=[128, 256]
            ),
            activation="relu"
        ),

        layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2, 0.5]
            )
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam", "sgd", "rmsprop"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [77]:
tuner = kt.RandomSearch(
    hypermodel=deep_build_cnn,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="deep_cnn"
)

In [78]:
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 21m 23s]
val_accuracy: 0.5908183455467224

Best val_accuracy So Far: 0.6693280339241028
Total elapsed time: 02h 08m 24s


In [79]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'filters1': 64, 'filters2': 128, 'filters3': 128, 'dense_units': 128, 'dropout': 0.2, 'optimizer': 'sgd'}


In [ ]:
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Epoch 1/5
208/220 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.3998 - loss: 1.7400

In [18]:
best_hp = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(7,activation='softmax')

])

best_hp.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 64)   │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 222, 222, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 128)  │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 109, 109, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,300,807 (43.11 MB)

 Trainable params: 11,300,167 (43.11 MB)

 Non-trainable params: 640 (2.50 KB)

In [19]:
best_hp.compile(optimizer='sgd',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [20]:
best_hp.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 613s 3s/step - accuracy: 0.2903 - loss: 2.3102 - val_accuracy: 0.0120 - val_loss: 2.8567
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 640s 3s/step - accuracy: 0.5248 - loss: 1.8365 - val_accuracy: 0.0699 - val_loss: 2.4175
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 683s 3s/step - accuracy: 0.4631 - loss: 1.7332 - val_accuracy: 0.0479 - val_loss: 4.7516
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 603s 3s/step - accuracy: 0.5113 - loss: 1.8136 - val_accuracy: 0.5502 - val_loss: 1.8218
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 615s 3s/step - accuracy: 0.5611 - loss: 1.7666 - val_accuracy: 0.4271 - val_loss: 1.9148


In [22]:
test_loss, test_accuracy = best_hp.evaluate(test_dataset)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 557ms/step - accuracy: 0.4258 - loss: 1.9219
Test Accuracy : 0.42581504583358765
Test Loss : 1.9219145774841309


In [ ]:
'''batch normalization'''

In [30]:
bn_cnn_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_cnn_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [31]:
bn_cnn_model.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [35]:
bn_cnn_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 329s 1s/step - accuracy: 0.2035 - loss: 8.5790 - val_accuracy: 0.0419 - val_loss: 1.9802
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.0937 - loss: 1.8593 - val_accuracy: 0.0612 - val_loss: 2.0586
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.4781 - loss: 1.7275 - val_accuracy: 0.5150 - val_loss: 1.9757
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 292s 1s/step - accuracy: 0.5396 - loss: 1.7727 - val_accuracy: 0.3839 - val_loss: 2.5291
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 302s 1s/step - accuracy: 0.5240 - loss: 1.7105 - val_accuracy: 0.5882 - val_loss: 1.7232
Restoring model weights from the end of the best epoch: 5.


In [36]:
loss, test_accuracy = bn_cnn_model.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 253ms/step - accuracy: 0.5921 - loss: 1.7076


In [37]:
bn_cnn_model.save(r'../models/bn_early_stop.keras')

In [ ]:
'''bn learning rate'''

In [58]:
bn_lr_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_lr_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [59]:
bn_lr_model.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [60]:
bn_lr_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict,callbacks=[lr_scheduler])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 288s 1s/step - accuracy: 0.2307 - loss: 7.1920 - val_accuracy: 0.0512 - val_loss: 3.2850 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 314s 1s/step - accuracy: 0.3725 - loss: 1.7828 - val_accuracy: 0.2974 - val_loss: 2.2945 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.4931 - loss: 1.7344 - val_accuracy: 0.5163 - val_loss: 1.8078 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 288s 1s/step - accuracy: 0.5389 - loss: 1.7025 - val_accuracy: 0.5110 - val_loss: 1.8197 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.5215 - loss: 1.7318 - val_accuracy: 0.5822 - val_loss: 1.6394 - learning_rate: 0.0010


In [65]:
test_loss,test_accuracy = bn_lr_model.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 239ms/step - accuracy: 0.5888 - loss: 1.6402


In [61]:
bn_lr_model.save(r'../models/bn_learningrate.keras')

In [ ]:
'''adam'''

In [70]:
bn_adam_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_adam_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [72]:
bn_adam_model.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [73]:
bn_adam_model.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 319s 1s/step - accuracy: 0.2463 - loss: 8.2743 - val_accuracy: 0.0126 - val_loss: 3.4291
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.1271 - loss: 1.7754 - val_accuracy: 0.0832 - val_loss: 2.2583
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.2037 - loss: 1.7550 - val_accuracy: 0.4817 - val_loss: 1.9546
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.5550 - loss: 1.6938 - val_accuracy: 0.4271 - val_loss: 2.0095
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 310s 1s/step - accuracy: 0.5801 - loss: 1.6589 - val_accuracy: 0.6174 - val_loss: 1.8527
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.5761 - loss: 1.6762 - val_accuracy: 0.5456 - val_loss: 1.7904
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.5513 - loss: 1.6189 - val_accuracy: 0.5343 - val_loss: 1.9820
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.5524 - loss: 1.6042 - val_accuracy: 0.596

In [74]:
test_loss, test_accuracy = bn_adam_model.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 235ms/step - accuracy: 0.5116 - loss: 1.7921


In [75]:
bn_adam_model.save(r'../models/bn_adam.keras')

In [ ]:
'''sgd'''

In [33]:
bn_SGD_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_SGD_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_14 (Activation)      │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [34]:
bn_SGD_model.compile(optimizer=tf.keras.optimizers.SGD(),loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [35]:
bn_SGD_model.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


220/220 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - accuracy: 0.3206 - loss: 2.1422 - val_accuracy: 0.0146 - val_loss: 3.3059
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 296s 1s/step - accuracy: 0.3096 - loss: 1.7238 - val_accuracy: 0.0120 - val_loss: 579.1065
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 266s 1s/step - accuracy: 0.3269 - loss: 3.0160 - val_accuracy: 0.3387 - val_loss: 2.0555
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 265s 1s/step - accuracy: 0.3398 - loss: 1.7624 - val_accuracy: 0.4697 - val_loss: 1.5897
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 291s 1s/step - accuracy: 0.4323 - loss: 1.6413 - val_accuracy: 0.5862 - val_loss: 1.1419
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.4400 - loss: 1.5945 - val_accuracy: 0.4977 - val_loss: 1.3593
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - accuracy: 0.4667 - loss: 1.5662 - val_accuracy: 0.2029 - val_loss: 1.7919
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.4175 - loss: 1.5184 - val_accuracy: 0.5

In [37]:
bn_SGD_model.save(r'../models/bn_sgd.keras')

In [ ]:
'''RMS PROP'''

In [38]:
bn_RMS_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_RMS_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_15 (Activation)      │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_17 (Activation)      │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [40]:
bn_RMS_model.compile(optimizer=tf.keras.optimizers.RMSprop(),loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [41]:
bn_RMS_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 305s 1s/step - accuracy: 0.2712 - loss: 11.1216 - val_accuracy: 0.0186 - val_loss: 2.4538
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.2364 - loss: 2.0553 - val_accuracy: 0.0446 - val_loss: 2.5752
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.3293 - loss: 1.9452 - val_accuracy: 0.6148 - val_loss: 1.4732
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 319s 1s/step - accuracy: 0.3991 - loss: 1.8632 - val_accuracy: 0.3287 - val_loss: 1.7184
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 332s 1s/step - accuracy: 0.3767 - loss: 1.7441 - val_accuracy: 0.4092 - val_loss: 1.5370


In [42]:
bn_RMS_model.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 283ms/step - accuracy: 0.4118 - loss: 1.5251


[1.5250855684280396, 0.4118429720401764]

In [43]:
bn_RMS_model.save(r'../models/bn_rms.keras')

In [ ]:
'''64'''

In [63]:
bn_64_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_64_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_29 (Conv2D)              │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_29 (Activation)      │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_30 (Activation)      │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_31 (Activation)      │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_31 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [64]:
bn_64_model.compile(optimizer='adam',loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [65]:
bn_64_model.fit(train_dataset_64,validation_data=val_dataset_64,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


110/110 ━━━━━━━━━━━━━━━━━━━━ 311s 3s/step - accuracy: 0.3217 - loss: 10.5252 - val_accuracy: 0.0858 - val_loss: 2.4568
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 304s 3s/step - accuracy: 0.3690 - loss: 1.9404 - val_accuracy: 0.0419 - val_loss: 3.5504
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 300s 3s/step - accuracy: 0.4049 - loss: 1.6028 - val_accuracy: 0.0246 - val_loss: 4.6236
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 283s 3s/step - accuracy: 0.4357 - loss: 1.5621 - val_accuracy: 0.1144 - val_loss: 2.8690
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 284s 3s/step - accuracy: 0.5139 - loss: 1.5770 - val_accuracy: 0.5263 - val_loss: 1.7501
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 288s 3s/step - accuracy: 0.5196 - loss: 1.5697 - val_accuracy: 0.4904 - val_loss: 1.9906
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 285s 3s/step - accuracy: 0.5218 - loss: 1.5066 - val_accuracy: 0.2116 - val_loss: 2.9775
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 663s 6s/step - accuracy: 0.5320 - loss: 1.5392 - val_accuracy: 0.55

In [66]:
bn_64_model.evaluate(test_dataset_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 14s 578ms/step - accuracy: 0.5236 - loss: 1.6494


[1.649423599243164, 0.5236194133758545]

In [67]:
bn_64_model.save(r'../models/bn_64.keras')

In [ ]:
'''hyper parameter'''

In [68]:
INPUT_SHAPE = (224, 224, 3)
NUM_CLASSES = 7

def bn_build_cnn(hp):

    model = models.Sequential([

        layers.Input(shape=INPUT_SHAPE),

        # Block 1
        layers.Conv2D(
            filters=hp.Choice(
                "filters1",
                values=[32, 64]
            ),
            kernel_size=(3,3),
            use_bias = False
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),

        # Block 2
        layers.Conv2D(
            filters=hp.Choice(
                "filters2",
                values=[64, 128]
            ),
            kernel_size=(3,3),
            use_bias=False,
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),

        # Block 3
        layers.Conv2D(
            filters=hp.Choice(
                "filters3",
                values=[128, 256]
            ),
            kernel_size=(3,3),
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),

        layers.Dense(
            units=hp.Choice(
                "dense_units",
                values=[128, 256]
            ),
            activation="relu"
        ),

        layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2, 0.5]
            )
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam", "sgd", "rmsprop"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [69]:
tuner = kt.RandomSearch(
    hypermodel=bn_build_cnn,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="bn_cnn"
)

In [70]:
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [01h 00m 50s]
val_accuracy: 0.038589488714933395

Best val_accuracy So Far: 0.6693280339241028
Total elapsed time: 03h 24m 54s


In [77]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'filters1': 32, 'filters2': 64, 'filters3': 256, 'dense_units': 128, 'dropout': 0.2, 'optimizer': 'rmsprop'}


In [79]:
best_model = tuner.get_best_models(num_models=1)[0]

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 15 variables. 
  saveable.load_own_variables(store)


In [80]:
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 413s 2s/step - accuracy: 0.2962 - loss: 1.9700 - val_accuracy: 0.6687 - val_loss: 1.9345
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 357s 2s/step - accuracy: 0.5620 - loss: 1.9465 - val_accuracy: 0.6687 - val_loss: 1.9316
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 428s 2s/step - accuracy: 0.6055 - loss: 1.9838 - val_accuracy: 0.6693 - val_loss: 1.9312
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 430s 2s/step - accuracy: 0.6173 - loss: 1.9465 - val_accuracy: 0.6693 - val_loss: 1.9307
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.5658 - loss: 1.9464 - val_accuracy: 0.6693 - val_loss: 1.9300


In [98]:
history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/bn_training_history.csv', index=False)

In [82]:
test_loss, test_accuracy = best_model.evaluate(test_dataset)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.6693 - loss: 1.9300
Test Accuracy : 0.6693280339241028
Test Loss : 1.9300243854522705


In [83]:
best_model.save(r'../models/best_deep.keras')

In [ ]:
'''drop out'''

In [88]:
dropout_cnn_model1 = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation='softmax')

])

dropout_cnn_model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [89]:
dropout_cnn_model1.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [92]:
history = dropout_cnn_model1.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 837ms/step - accuracy: 0.3440 - loss: 1.9400 - val_accuracy: 0.2681 - val_loss: 1.8303
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 186s 823ms/step - accuracy: 0.2196 - loss: 1.9187 - val_accuracy: 0.3626 - val_loss: 1.7111
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 826ms/step - accuracy: 0.3754 - loss: 1.8563 - val_accuracy: 0.5988 - val_loss: 1.3503
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 191s 845ms/step - accuracy: 0.2132 - loss: 1.9228 - val_accuracy: 0.1730 - val_loss: 1.8631
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 825ms/step - accuracy: 0.3513 - loss: 1.8119 - val_accuracy: 0.3706 - val_loss: 1.6906
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 816ms/step - accuracy: 0.3561 - loss: 1.8215 - val_accuracy: 0.3752 - val_loss: 1.7968
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [93]:
dropout_cnn_model1.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 216ms/step - accuracy: 0.5968 - loss: 1.3598


[1.359782338142395, 0.5968064069747925]

In [94]:
dropout_cnn_model1.save(r'../models/drop_earlystop.keras')

In [95]:
history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/bn_training_history.csv', index=False)